# Retail Orders: Data Profile & Quality Checks

**Decision owner:** Head of Growth & Revenue Operations (decides pricing, discounting and campaign spend from this data)
**Dataset:** `retail-orders-raw.csv` (intentionally imperfect order extract) + `retail-data-dictionary.csv`
**Purpose:** run executable checks for the five quality dimensions (completeness, uniqueness, validity, consistency, freshness), compare the results with the thresholds in `data_quality_contract.md`, then clean the data and compute the KPIs defined in `kpi_dictionary.xlsx`.

**How to run:** put this notebook in the same folder as the two CSV files and use *Run All*. Set `RUN_DATE` to `pd.Timestamp.today().normalize()` for a live run.

In [1]:
import re
import pandas as pd
from pathlib import Path
from IPython.display import display

RAW_PATH = Path("retail-orders-raw.csv")
DICT_PATH = Path("retail-data-dictionary.csv")

RUN_DATE = pd.Timestamp("2026-10-05")   # fixed so results are reproducible; use pd.Timestamp.today().normalize() for live runs
DATE_FLOOR = pd.Timestamp("2025-01-01") # from the data dictionary
FRESHNESS_SLA_DAYS = 2                  # contract: newest order must be at most 2 days old

ALLOWED_CATEGORY = {"Learning Kit", "Course Access", "Mentor Session"}
ALLOWED_SEGMENT = {"Student", "Fresher", "Professional"}
ALLOWED_STATUS = {"Paid", "Pending", "Failed", "Refunded"}

raw = pd.read_csv(RAW_PATH, dtype=str)   # read everything as text so nothing is silently coerced
dictionary = pd.read_csv(DICT_PATH)
N = len(raw)
print(f"{N} rows x {raw.shape[1]} columns")
display(dictionary)
display(raw)

12 rows x 9 columns


,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0,Paid


## 1. Quick profile

In [2]:
profile = pd.DataFrame({
    "non_null": raw.notna().sum(),
    "nulls": raw.isna().sum(),
    "distinct": raw.nunique(),
    "sample_values": raw.apply(lambda s: ", ".join(s.dropna().unique()[:4])),
})
display(profile)

,non_null,nulls,distinct,sample_values
order_id,12,0,11,"RT-1001, RT-1002, RT-1003, RT-1004"
order_date,11,1,10,"2026-01-03, 03/01/2026, 2026-01-05, 2026-01-07"
customer_segment,12,0,4,"Student, Fresher, student, Professional"
city,11,1,7,"Chennai, Bengaluru, Hyderabad, Pune"
category,12,0,3,"Learning Kit, Course Access, Mentor Session"
quantity,12,0,5,"2, 1, 3, -1"
unit_price,12,0,3,"799, 1499, 999"
discount_pct,11,1,5,"10, 0, 5, 105"
payment_status,12,0,5,"Paid, paid, Pending, Failed"


## 2. Check framework

Every check records: how many rows fail, the failure rate, the contract threshold, and PASS/FAIL. Thresholds are the *maximum tolerated failure rate* (0 = zero tolerance). Severity (P1/P2/P3) drives the escalation action in the contract.

In [3]:
results = []

def record(cid, dim, rule, column, failed, total, threshold, severity, unit="rate"):
    metric = failed / total if unit == "rate" else failed
    results.append({
        "check_id": cid, "dimension": dim, "rule": rule, "column": column,
        "rows_failed": failed if unit == "rate" else None,
        "metric": round(metric, 4), "unit": unit,
        "threshold": threshold, "severity": severity,
        "status": "PASS" if metric <= threshold else "FAIL",
    })

num = lambda col: pd.to_numeric(raw[col], errors="coerce")
blank = lambda col: raw[col].isna() | (raw[col].str.strip() == "")

### 2.1 Completeness

In [4]:
record("C1", "Completeness", "order_date is not null", "order_date", int(blank("order_date").sum()), N, 0.00, "P1")
record("C2", "Completeness", "city is not null", "city", int(blank("city").sum()), N, 0.02, "P2")
record("C3", "Completeness", "discount_pct is not null (missing is not assumed to be 0)", "discount_pct", int(blank("discount_pct").sum()), N, 0.00, "P2")
other_cols = ["order_id", "customer_segment", "category", "quantity", "unit_price", "payment_status"]
record("C4", "Completeness", "all other required columns are not null", ", ".join(other_cols),
       int(raw[other_cols].isna().any(axis=1).sum()), N, 0.00, "P1")

### 2.2 Uniqueness

In [5]:
record("U1", "Uniqueness", "order_id appears once", "order_id", int(raw.duplicated("order_id").sum()), N, 0.00, "P1")
record("U2", "Uniqueness", "no fully duplicated rows", "all columns", int(raw.duplicated().sum()), N, 0.00, "P1")

### 2.3 Validity

In [6]:
iso = raw["order_date"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
parsed_iso = pd.to_datetime(raw["order_date"].where(iso), format="%Y-%m-%d", errors="coerce")
in_range = parsed_iso.between(DATE_FLOOR, RUN_DATE)
record("V1", "Validity", "ISO-formatted order_date is a real date between 2025-01-01 and today", "order_date",
       int((iso & ~in_range).sum()), N, 0.00, "P1")

q = num("quantity")
record("V2", "Validity", "quantity is a whole number > 0", "quantity",
       int((~((q > 0) & (q % 1 == 0))).sum()), N, 0.00, "P1")

p = num("unit_price")
record("V3", "Validity", "unit_price is a non-negative number", "unit_price", int((~(p >= 0)).sum()), N, 0.00, "P1")

d = num("discount_pct")
record("V4", "Validity", "discount_pct is between 0 and 100", "discount_pct",
       int((d.notna() & ~d.between(0, 100)).sum()), N, 0.00, "P1")

record("V5", "Validity", "category is one of the 3 allowed products", "category",
       int((~raw["category"].isin(ALLOWED_CATEGORY)).sum()), N, 0.00, "P2")

seg_ok = raw["customer_segment"].str.strip().str.title().isin(ALLOWED_SEGMENT)
sta_ok = raw["payment_status"].str.strip().str.title().isin(ALLOWED_STATUS)
record("V6", "Validity", "segment and payment_status map to an allowed value after normalization",
       "customer_segment, payment_status", int((~(seg_ok & sta_ok)).sum()), N, 0.00, "P2")

### 2.4 Consistency

In [7]:
record("K1", "Consistency", "customer_segment uses canonical casing", "customer_segment",
       int((raw["customer_segment"] != raw["customer_segment"].str.strip().str.title()).sum()), N, 0.00, "P3")
record("K2", "Consistency", "payment_status uses canonical casing", "payment_status",
       int((raw["payment_status"] != raw["payment_status"].str.strip().str.title()).sum()), N, 0.00, "P3")
record("K3", "Consistency", "order_date uses one format (YYYY-MM-DD)", "order_date",
       int((raw["order_date"].notna() & ~iso).sum()), N, 0.00, "P2")

price_variants = raw.assign(price=p).groupby("category")["price"].transform("nunique")
record("K4", "Consistency", "each category has a single unit_price", "category, unit_price", int((price_variants > 1).sum()), N, 0.00, "P2")

conflict_ids = raw.drop_duplicates().groupby("order_id").size()
record("K5", "Consistency", "no order_id has conflicting versions", "order_id", int((conflict_ids > 1).sum()), N, 0.00, "P1")

### 2.5 Freshness

In [8]:
latest = parsed_iso.max()
days_stale = (RUN_DATE - latest).days
print(f"Latest valid order_date: {latest.date()} | run date: {RUN_DATE.date()} | age: {days_stale} days")
record("F1", "Freshness", f"newest order is at most {FRESHNESS_SLA_DAYS} days old", "order_date", days_stale, None, FRESHNESS_SLA_DAYS, "P1", unit="days")

Latest valid order_date: 2026-01-18 | run date: 2026-10-05 | age: 260 days


## 3. Results vs contract

In [9]:
dq = pd.DataFrame(results)
display(dq)
print()
print(dq.groupby(["severity", "status"]).size().unstack(fill_value=0))

p1_fail = dq[(dq.severity == "P1") & (dq.status == "FAIL")]
verdict = "BLOCK: do not publish dashboards from this extract" if len(p1_fail) else "PASS"
print(f"\nContract verdict: {verdict} ({len(p1_fail)} P1 checks failed: {', '.join(p1_fail.check_id)})")
dq.to_csv("dq_results.csv", index=False)

,check_id,dimension,rule,column,rows_failed,metric,unit,threshold,severity,status
0,C1,Completeness,order_date is not null,order_date,1.0,0.0833,rate,0.00,P1,FAIL
1,C2,Completeness,city is not null,city,1.0,0.0833,rate,0.02,P2,FAIL
2,C3,Completeness,discount_pct is not null (missing is not assum...,discount_pct,1.0,0.0833,rate,0.00,P2,FAIL
3,C4,Completeness,all other required columns are not null,"order_id, customer_segment, category, quantity...",0.0,0.0000,rate,0.00,P1,PASS
4,U1,Uniqueness,order_id appears once,order_id,1.0,0.0833,rate,0.00,P1,FAIL
5,U2,Uniqueness,no fully duplicated rows,all columns,1.0,0.0833,rate,0.00,P1,FAIL
6,V1,Validity,ISO-formatted order_date is a real date betwee...,order_date,1.0,0.0833,rate,0.00,P1,FAIL
7,V2,Validity,quantity is a whole number > 0,quantity,2.0,0.1667,rate,0.00,P1,FAIL
8,V3,Validity,unit_price is a non-negative number,unit_price,0.0,0.0000,rate,0.00,P1,PASS
9,V4,Validity,discount_pct is between 0 and 100,discount_pct,1.0,0.0833,rate,0.00,P1,FAIL



status    FAIL  PASS
severity            
P1           7     3
P2           3     3
P3           2     0

Contract verdict: BLOCK: do not publish dashboards from this extract (7 P1 checks failed: C1, U1, U2, V1, V2, V4, F1)


## 4. Remediation: dedupe, normalize, quarantine

Rules (also listed in the contract):
1. Exact duplicate rows are dropped (first kept).
2. Segment and status casing is normalized.
3. `DD/MM/YYYY` dates are repaired to ISO (India date convention; flagged as a repair).
4. Rows with a **blocking defect** (missing or invalid date, invalid quantity, missing or out-of-range discount) go to quarantine and are excluded from every KPI. Nothing is guessed (e.g. `"two"` is not converted to 2, a missing discount is not assumed to be 0).
5. Missing `city` is non-blocking: the row stays in revenue KPIs and is flagged.

In [10]:
df = raw.copy()
dup_mask = df.duplicated(keep="first")
dropped_duplicates = df[dup_mask]
df = df[~dup_mask].copy()

df["customer_segment"] = df["customer_segment"].str.strip().str.title()
df["payment_status"] = df["payment_status"].str.strip().str.title()

def parse_date(s):
    if pd.isna(s) or not str(s).strip():
        return pd.NaT, "missing order_date", None
    s = str(s).strip()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        t = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
        return (t, None, None) if pd.notna(t) else (pd.NaT, f"invalid calendar date '{s}'", None)
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s):
        t = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
        return (t, None, "date repaired from DD/MM/YYYY") if pd.notna(t) else (pd.NaT, f"invalid calendar date '{s}'", None)
    return pd.NaT, f"unrecognised date format '{s}'", None

clean_rows, quarantine_rows = [], []
for _, r in df.iterrows():
    reasons, notes = [], []
    dt, why, note = parse_date(r["order_date"])
    if why: reasons.append(why)
    if note: notes.append(note)
    if pd.notna(dt) and not (DATE_FLOOR <= dt <= RUN_DATE): reasons.append(f"order_date {dt.date()} outside allowed range")

    qty = pd.to_numeric(r["quantity"], errors="coerce")
    if pd.isna(qty): reasons.append(f"quantity not numeric ('{r['quantity']}')")
    elif qty <= 0 or qty % 1: reasons.append(f"quantity must be a whole number > 0 ({r['quantity']})")

    price = pd.to_numeric(r["unit_price"], errors="coerce")
    if pd.isna(price) or price < 0: reasons.append(f"invalid unit_price ({r['unit_price']})")

    disc = pd.to_numeric(r["discount_pct"], errors="coerce")
    if pd.isna(disc): reasons.append("discount_pct missing (no justification on file)")
    elif not 0 <= disc <= 100: reasons.append(f"discount_pct out of range ({r['discount_pct']})")

    if r["category"] not in ALLOWED_CATEGORY: reasons.append(f"unknown category ({r['category']})")
    if r["customer_segment"] not in ALLOWED_SEGMENT: reasons.append(f"unknown segment ({r['customer_segment']})")
    if r["payment_status"] not in ALLOWED_STATUS: reasons.append(f"unknown payment_status ({r['payment_status']})")
    if pd.isna(r["city"]) or not str(r["city"]).strip(): notes.append("city missing (non-blocking)")

    if reasons:
        quarantine_rows.append({**raw.loc[r.name].to_dict(), "source_row": r.name + 2, "reject_reason": "; ".join(reasons)})
    else:
        clean_rows.append({
            "order_id": r["order_id"], "order_date": dt.date().isoformat(), "customer_segment": r["customer_segment"],
            "city": r["city"], "category": r["category"], "quantity": int(qty), "unit_price": float(price),
            "discount_pct": float(disc), "payment_status": r["payment_status"], "repair_note": "; ".join(notes),
        })

clean = pd.DataFrame(clean_rows)
quarantine = pd.DataFrame(quarantine_rows)
assert len(clean) + len(quarantine) + len(dropped_duplicates) == N, "row accounting mismatch"
print(f"raw rows {N} = clean {len(clean)} + quarantined {len(quarantine)} + duplicates dropped {len(dropped_duplicates)}")
display(clean)
display(quarantine[["source_row", "order_id", "reject_reason"]])
clean.to_csv("clean_orders.csv", index=False)
quarantine.to_csv("quarantine_orders.csv", index=False)

raw rows 12 = clean 6 + quarantined 5 + duplicates dropped 1


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,repair_note
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799.0,10.0,Paid,
1,RT-1002,2026-01-03,Fresher,Bengaluru,Course Access,1,1499.0,0.0,Paid,date repaired from DD/MM/YYYY
2,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799.0,5.0,Paid,
3,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999.0,0.0,Failed,city missing (non-blocking)
4,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799.0,0.0,Paid,
5,RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499.0,15.0,Refunded,


,source_row,order_id,reject_reason
0,4,RT-1003,discount_pct missing (no justification on file)
1,8,RT-1006,invalid calendar date '2026-13-10'; quantity m...
2,9,RT-1007,discount_pct out of range (105)
3,10,RT-1008,quantity not numeric ('two')
4,13,RT-1011,missing order_date


## 5. KPI calculation on the clean set

Formulas match `kpi_dictionary.xlsx` (sheet *KPI_Dictionary*). The workbook recalculates the same numbers with live formulas on its *Clean_Data* sheet, which is a second, independent check.

In [11]:
c = clean.copy()
c["gross"] = c["quantity"] * c["unit_price"]
c["discount_amt"] = c["gross"] * c["discount_pct"] / 100
c["net"] = c["gross"] - c["discount_amt"]

booked = c[c.payment_status != "Failed"]
paid = c[c.payment_status == "Paid"]
distinct_raw_orders = raw["order_id"].nunique()

kpis = {
    "KPI-01 Order Count": len(c),
    "KPI-02 Gross Merchandise Value (INR)": booked["gross"].sum(),
    "KPI-03 Discount Amount (INR)": booked["discount_amt"].sum(),
    "KPI-04 Discount Rate": booked["discount_amt"].sum() / booked["gross"].sum(),
    "KPI-05 Net Revenue (INR)": paid["net"].sum(),
    "KPI-06 Average Order Value (INR)": paid["net"].sum() / len(paid),
    "KPI-07 Units per Order": paid["quantity"].sum() / len(paid),
    "KPI-08 Payment Success Rate": (c.payment_status == "Paid").sum() / c.payment_status.isin(["Paid", "Failed"]).sum(),
    "KPI-09 Refund Rate": (c.payment_status == "Refunded").sum() / c.payment_status.isin(["Paid", "Refunded"]).sum(),
    "KPI-10 Data Quality Pass Rate": len(c) / distinct_raw_orders,
}
kpi_df = pd.DataFrame({"kpi": kpis.keys(), "value_on_clean_sample": [round(v, 4) for v in kpis.values()]})
display(kpi_df)
kpi_df.to_csv("kpi_values.csv", index=False)

,kpi,value_on_clean_sample
0,KPI-01 Order Count,6.0000
1,KPI-02 Gross Merchandise Value (INR),9291.0000
2,KPI-03 Discount Amount (INR),729.3500
3,KPI-04 Discount Rate,0.0785
4,KPI-05 Net Revenue (INR),6013.3500
5,KPI-06 Average Order Value (INR),1503.3375
6,KPI-07 Units per Order,1.7500
7,KPI-08 Payment Success Rate,0.8000
8,KPI-09 Refund Rate,0.2000
9,KPI-10 Data Quality Pass Rate,0.5455


## 6. Findings

* The raw extract fails **12 of 18** checks (6 pass). Failures include 1 duplicate order, 2 invalid quantities, 1 out-of-range discount, 1 impossible date, 1 non-ISO date, 3 missing values (date, city, discount) and 2 casing inconsistencies (segment, payment status).
* **Freshness fails badly:** the newest valid order is from 2026-01-18, which is far older than the 2-day SLA, so a live dashboard fed by this extract would show stale numbers.
* Only **6 of 11 distinct orders (55%)** are trustworthy enough for revenue KPIs. Reporting on the raw file would mis-state revenue (the duplicate alone inflates it, and a 105% discount produces negative revenue).
* Per the contract, any P1 failure blocks publication until the data owner fixes the source or signs a documented exception.